## 🧠 PASO 8 — Red Neuronal MLP con PyTorch
Arquitectura: [128 → 64 → 32] con BatchNorm, ReLU y Dropout(0.3)

In [ ]:
# ── Definición del MLP ────────────────────────────────────────
class MLP(nn.Module):
    """Perceptrón Multicapa para clasificación binaria."""
    def __init__(self, in_dim=16, hidden=[128, 64, 32], dropout=0.3):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden:
            layers += [
                nn.Linear(prev, h),
                nn.BatchNorm1d(h),
                nn.ReLU(),
                nn.Dropout(dropout)
            ]
            prev = h
        layers.append(nn.Linear(prev, 1))   # Salida escalar (logit)
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(1)

# ── Preparación de datos para PyTorch ─────────────────────────
ds_train = TensorDataset(torch.tensor(Xsm), torch.tensor(ysm))
loader   = DataLoader(ds_train, batch_size=256, shuffle=True)
Xv_t     = torch.tensor(Xv_s)
Xte_t    = torch.tensor(Xte_s)

# ── Entrenamiento ─────────────────────────────────────────────
EPOCHS = 60
mlp    = MLP()
crit   = nn.BCEWithLogitsLoss()
opt    = torch.optim.Adam(mlp.parameters(), lr=1e-3, weight_decay=1e-4)
sch    = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, patience=5, factor=0.5)

tr_losses, vl_losses = [], []
best_auc, best_state = 0, None

print(f'Entrenando MLP durante {EPOCHS} épocas...')
for ep in range(EPOCHS):
    mlp.train()
    ep_loss = []
    for bx, by in loader:
        opt.zero_grad()
        loss = crit(mlp(bx), by)
        loss.backward(); opt.step()
        ep_loss.append(loss.item())

    mlp.eval()
    with torch.no_grad():
        vo   = mlp(Xv_t)
        vl   = crit(vo, torch.tensor(y_val, dtype=torch.float32)).item()
        vp   = torch.sigmoid(vo).numpy()
        vauc = roc_auc_score(y_val, vp)

    sch.step(vl)
    tr_losses.append(np.mean(ep_loss))
    vl_losses.append(vl)

    if vauc > best_auc:
        best_auc   = vauc
        best_state = {k: v.clone() for k, v in mlp.state_dict().items()}

    if (ep + 1) % 20 == 0:
        print(f'  Época {ep+1:3d}: '
              f'train_loss={tr_losses[-1]:.4f}  '
              f'val_loss={vl:.4f}  '
              f'val_AUC={vauc:.4f}')

mlp.load_state_dict(best_state)
print(f'\n✓ Mejor AUC-ROC en validación: {best_auc:.4f}')

# ── Evaluación en prueba ───────────────────────────────────────
mlp.eval()
with torch.no_grad():
    mlp_probs = torch.sigmoid(mlp(Xte_t)).numpy()
mlp_preds = (mlp_probs > 0.5).astype(int)

print(f'\n=== MLP — Conjunto de prueba ===')
print(f'  AUC-ROC:  {roc_auc_score(y_test, mlp_probs):.4f}')
print(f'  F1-macro: {f1_score(y_test, mlp_preds, average="macro"):.4f}')
print(classification_report(y_test, mlp_preds,
      target_names=['No readmit.', 'Readmitido']))

## 📈 PASO 9 — Comparación de modelos y visualizaciones

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Curvas ROC ────────────────────────────────────────────────
ax = axes[0]
lr_probs  = lr.predict_proba(Xte_s)[:, 1]
rf_probs  = rf.predict_proba(Xte_s)[:, 1]
svm_probs = svm.predict_proba(Xte_s)[:, 1]

for nombre, probs, col in [
    ('Reg. Logística', lr_probs,  C['p']),
    ('Random Forest',  rf_probs,  C['s']),
    ('SVM (RBF)',      svm_probs, C['a']),
    ('MLP [128-64-32]',mlp_probs, C['r'])
]:
    fpr, tpr, _ = roc_curve(y_test, probs)
    auc = roc_auc_score(y_test, probs)
    ax.plot(fpr, tpr, color=col, lw=2, label=f'{nombre} (AUC={auc:.4f})')

ax.plot([0,1],[0,1], '--', color='gray', lw=1.2, label='Aleatorio (AUC=0.50)')
ax.set_xlabel('Tasa de Falsos Positivos (FPR)', fontsize=11)
ax.set_ylabel('Tasa de Verdaderos Positivos (TPR)', fontsize=11)
ax.set_title('Curvas ROC — Comparación de modelos', fontsize=12,
              fontweight='bold', color=C['p'])
ax.legend(fontsize=9); ax.grid(alpha=0.3)

# ── Convergencia del MLP ───────────────────────────────────────
ax2 = axes[1]
ax2.plot(range(1, EPOCHS+1), tr_losses, color=C['p'], lw=2,
         label='Pérdida entrenamiento')
ax2.plot(range(1, EPOCHS+1), vl_losses, color=C['s'], lw=2, ls='--',
         label='Pérdida validación')
ax2.set_xlabel('Época', fontsize=11); ax2.set_ylabel('BCE Loss', fontsize=11)
ax2.set_title('Curva de convergencia del MLP [128-64-32]',
               fontsize=12, fontweight='bold', color=C['p'])
ax2.legend(fontsize=10); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('comparacion_modelos.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Tabla resumen de métricas ─────────────────────────────────
todos = dict(resultados_ml)
todos['MLP [128-64-32]'] = {
    'AUC-ROC':   round(roc_auc_score(y_test, mlp_probs), 4),
    'F1-macro':  round(f1_score(y_test, mlp_preds, average='macro'), 4),
    'Precisión': round(precision_score(y_test, mlp_preds), 4),
    'Recall':    round(recall_score(y_test, mlp_preds), 4),
    'Accuracy':  round((mlp_preds == y_test).mean(), 4),
}
df_res = pd.DataFrame(todos).T
print('\n=== Tabla resumen de métricas en el conjunto de prueba ===')
display(df_res.style.highlight_max(axis=0, color='#A8D1F0'))